In [1]:
import pandas as pd

In [2]:
url = "https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2025-10.parquet"
columns = ['lpep_pickup_datetime', 'lpep_dropoff_datetime', 'PULocationID', 'DOLocationID', 'passenger_count', 'trip_distance', 'tip_amount', 'total_amount']

In [3]:
df = pd.read_parquet(url, columns=columns)
df.head()

,lpep_pickup_datetime,lpep_dropoff_datetime,PULocationID,DOLocationID,passenger_count,trip_distance,tip_amount,total_amount
0,2025-10-01 00:21:47,2025-10-01 00:24:37,247,69,1.0,0.70,1.70,10.00
1,2025-10-01 00:14:03,2025-10-01 00:24:14,66,25,1.0,1.61,2.78,16.68
2,2025-10-01 00:16:44,2025-10-01 00:16:47,244,244,1.0,0.00,2.20,13.20
3,2025-10-01 00:07:36,2025-10-01 00:32:14,95,170,1.0,10.37,11.31,67.85
4,2025-09-30 21:10:29,2025-09-30 21:22:30,82,138,1.0,4.07,6.82,34.12


In [4]:
row = df.iloc[0]
row

lpep_pickup_datetime     2025-10-01 00:21:47
lpep_dropoff_datetime    2025-10-01 00:24:37
PULocationID                             247
DOLocationID                              69
passenger_count                          1.0
trip_distance                            0.7
tip_amount                               1.7
total_amount                            10.0
Name: 0, dtype: object

In [5]:
from models import Ride, ride_from_row, ride_serializer

In [6]:
green_ride = ride_from_row(df.iloc[1])
green_ride

Ride(lpep_pickup_datetime='2025-10-01 00:14:03', lpep_dropoff_datetime='2025-10-01 00:24:14', PULocationID=66, DOLocationID=25, passenger_count=1.0, trip_distance=1.61, tip_amount=2.78, total_amount=16.68)

In [7]:
from kafka import KafkaProducer

server = 'localhost:9092'

producer = KafkaProducer(
    bootstrap_servers=[server],
    value_serializer=ride_serializer
)

In [8]:
topic_name = 'green_trips'

In [9]:
print(df.columns)

Index(['lpep_pickup_datetime', 'lpep_dropoff_datetime', 'PULocationID',
       'DOLocationID', 'passenger_count', 'trip_distance', 'tip_amount',
       'total_amount'],
      dtype='str')


In [ ]:
from time import time, sleep

t0 = time()

for _, row in df.iterrows():
    ride = ride_from_row(row)
    producer.send(topic_name, value=ride)
    print(f"Sent: {ride}")
    sleep(0.01)

producer.flush()

t1 = time()
print(f'took {(t1 - t0):.2f} seconds')